the next step is DEGs (Differentially Expressed Genes) per niche.

What DEGs means here: For each of the 10 HC Ward niches, we find which genes are significantly upregulated or downregulated compared to all other niches combined. This tells us the gene-level biological identity of each niche, going deeper than cell type compositions into actual molecular signatures.


First check barcode matching, if they match we continue

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np

# Load both files
adata_niches = sc.read_h5ad(
    "/path/to/data/"
    "Cell2location/spatial_mapping/exploration_mean/"
    "spatial_compositional_space_mean.h5ad"
)

adata_gex = sc.read_h5ad(
    "/path/to/data/"
    "Analyses_Erickson_Junyi/Thesis_QC_C2L_Ready.h5ad"
)

print("Niche adata shape:", adata_niches.shape)
print("GEX adata shape:", adata_gex.shape)

# Check index format in both
print("\nNiche barcodes (first 5):", adata_niches.obs_names[:5].tolist())
print("GEX barcodes (first 5):", adata_gex.obs_names[:5].tolist())

# Check overlap
overlap = adata_niches.obs_names.isin(adata_gex.obs_names).sum()
print(f"\nOverlapping barcodes: {overlap}")
print(f"Only in niches: {(~adata_niches.obs_names.isin(adata_gex.obs_names)).sum()}")
print(f"Only in GEX: {(~adata_gex.obs_names.isin(adata_niches.obs_names)).sum()}")

# Check layers
print("\nGEX layers:", list(adata_gex.layers.keys()))
print("GEX obs columns:", adata_gex.obs.columns.tolist())

In [ ]:
import scipy.sparse as sp

X = adata_gex.X
if sp.issparse(X):
    sample_vals = X.data[:100]
else:
    sample_vals = X.flatten()[:100]

print("Sample values from X (should be integers like 0,1,2,3... if raw):")
print(sample_vals)
print("\nMax value:", X.max())
print("Are all values integers?", np.all(sample_vals == sample_vals.astype(int)))
print("Any values > 20?", (sample_vals > 20).any())

We then use Thesis_Raw_Combined_Master.h5ad instead of Thesis_QC_C2L_Ready.h5ad as this one doesnt actually have raw counts

In [ ]:
# ── Cell 1 — Load and transfer niche labels ──────────────────────────

import os
import scanpy as sc
import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt

OUT_DIR = ("/path/to/data/"
           "Cell2location/spatial_mapping/exploration_mean/DEGs")
os.makedirs(OUT_DIR, exist_ok=True)

CLUSTER_KEY = 'hc_ward_k10'

# Load niche labels
adata_niches = sc.read_h5ad(
    "/path/to/data/"
    "Cell2location/spatial_mapping/exploration_mean/"
    "spatial_compositional_space_mean.h5ad"
)

# Load RAW master file — this has true integer counts
adata_raw = sc.read_h5ad(
    "/path/to/data/"
    "Analyses_Erickson_Junyi/Thesis_Raw_Combined_Master.h5ad"
)

print("Niche adata shape:", adata_niches.shape)
print("Raw adata shape:", adata_raw.shape)
print("\nNiche barcodes (first 3):", adata_niches.obs_names[:3].tolist())
print("Raw barcodes (first 3):", adata_raw.obs_names[:3].tolist())

# Check overlap
overlap = adata_niches.obs_names.isin(adata_raw.obs_names).sum()
print(f"\nOverlapping barcodes: {overlap}")
print(f"Only in niches: {(~adata_niches.obs_names.isin(adata_raw.obs_names)).sum()}")
print(f"Only in raw: {(~adata_raw.obs_names.isin(adata_niches.obs_names)).sum()}")

In [ ]:
# Subset raw to only the 101,490 spots that have niche labels
adata_raw = adata_raw[adata_niches.obs_names].copy()
print("After subset:", adata_raw.shape)

# Verify raw counts are actually integers
X = adata_raw.X
sample_vals = X.data[:20] if sp.issparse(X) else X.flatten()[:20]
print("\nSample values (should be integers):", sample_vals)
print("Are integers:", np.all(sample_vals == sample_vals.astype(int)))

# Transfer niche labels
adata_raw.obs[CLUSTER_KEY] = adata_niches.obs[CLUSTER_KEY].values
print("\nNiche distribution:")
print(adata_raw.obs[CLUSTER_KEY].value_counts().sort_index())

In [ ]:
# Save raw counts before normalisation
adata_raw.layers['counts'] = adata_raw.X.copy()

# Normalise for DEG analysis
sc.pp.normalize_total(adata_raw, target_sum=1e4)
sc.pp.log1p(adata_raw)

# Verify
X_check = adata_raw.X
sample_check = X_check.data[:5] if sp.issparse(X_check) else X_check.flatten()[:5]
print("Post-normalisation sample values:", sample_check)
print("Max value:", adata_raw.X.max())

# Run DEGs
print("\nRunning DEGs — this may take 5-10 minutes...")
sc.tl.rank_genes_groups(
    adata_raw,
    groupby=CLUSTER_KEY,
    method='wilcoxon',
    key_added='rank_genes_hc_ward_k10',
    pts=True,
    n_genes=200
)
print("Done.")

In [ ]:
results = adata_raw.uns['rank_genes_hc_ward_k10']
niches = results['names'].dtype.names

all_degs = []
for niche in niches:
    df = pd.DataFrame({
        'gene':          results['names'][niche],
        'score':         results['scores'][niche],
        'pval':          results['pvals'][niche],
        'pval_adj':      results['pvals_adj'][niche],
        'logfoldchange': results['logfoldchanges'][niche],
    })
    df['niche'] = niche
    df = df[df['pval_adj'] < 0.05]
    df = df[df['logfoldchange'] > 0]
    all_degs.append(df)

degs_df = pd.concat(all_degs, ignore_index=True)
degs_df.to_csv(os.path.join(OUT_DIR, 'DEGs_hc_ward_k10.csv'), index=False)

print(f"Total significant upregulated DEGs: {len(degs_df)}")
print("\nTop 5 DEGs per niche:")
print(degs_df.groupby('niche').head(5)[
    ['niche', 'gene', 'logfoldchange', 'pval_adj']
].to_string(index=False))

Check 1 — the suggested non-ribosomal DEGs for niche 6:

In [ ]:
niche6_degs = degs_df[degs_df['niche'] == '6']
niche6_no_ribo = niche6_degs[~niche6_degs['gene'].str.startswith(('RPS', 'RPL'))]
print("Top 15 non-ribosomal DEGs for niche 6:")
print(niche6_no_ribo.head(15)[['gene', 'logfoldchange', 'pval_adj']].to_string(index=False))

Check 2 — QC metrics across all suspicious niches:


In [ ]:
for niche in ['6', '7', '8']:
    mask = adata_raw.obs[CLUSTER_KEY] == niche
    print(f"\nNiche {niche} QC metrics:")
    print(adata_raw.obs.loc[mask,
          ['n_genes_by_counts', 'total_counts', 'pct_counts_mt']
          ].describe().round(2))

we think niche 6 is inmune based, check:

In [ ]:
# Look at raw results before any filtering
results = adata_raw.uns['rank_genes_hc_ward_k10']

# Get ALL 200 genes for niche 6
niche6_full = pd.DataFrame({
    'gene':          results['names']['6'],
    'logfoldchange': results['logfoldchanges']['6'],
    'pval_adj':      results['pvals_adj']['6'],
    'score':         results['scores']['6'],
})
niche6_full['rank'] = range(1, len(niche6_full) + 1)

# Search for canonical immune markers regardless of filtering
canonical_immune = ['JCHAIN', 'IGHG1', 'IGHG2', 'IGHG3', 'IGHG4',
                    'MS4A1', 'CD19', 'CD79A', 'CD79B',
                    'CD8A', 'CD8B', 'GZMB', 'GZMK', 'PRF1',
                    'FOXP3', 'IL2RA', 'CTLA4',
                    'MZB1', 'XBP1', 'PRDM1',
                    'CD3D', 'CD3E', 'CD3G']

found = niche6_full[niche6_full['gene'].isin(canonical_immune)]
print("Canonical immune markers in niche 6 (unfiltered, all 200 genes):")
print(found[['rank', 'gene', 'logfoldchange', 'pval_adj']].to_string(index=False))

print(f"\nTotal genes in niche 6 results: {len(niche6_full)}")
print("\nBottom 10 genes (lowest ranked):")
print(niche6_full.tail(10)[['rank', 'gene', 'logfoldchange']].to_string(index=False))

In [ ]:
missing = [g for g in ['JCHAIN', 'MS4A1', 'CD79A', 'CD8A', 'GZMB'] 
           if g not in adata_raw.var_names]
print("Markers completely absent from dataset:", missing)

 markers exist in the dataset but don't appear in the top 200 DEGs for niche 6. This means they are not differentially expressed in niche 6 compared to all other niches combined in a one-vs-rest test.
This doesn't mean niche 6 isn't immune-enriched — it means the one-vs-rest comparison is diluting the signal. When you compare niche 6 against all 9 other niches combined, many of those other niches also contain some B cells and T cells, so the immune genes don't appear uniquely upregulated in niche 6.

Check absolute expression levels instead
The question isn't whether these genes are differentially expressed — it's whether they are highly expressed in niche 6 compared to other niches.


In [ ]:
import scipy.sparse as sp

# Get raw normalised expression for immune markers
markers_to_check = ['JCHAIN', 'MS4A1', 'CD79A', 'CD8A', 
                    'GZMB', 'CD19', 'MZB1', 'XBP1']

# Build expression dataframe
X_norm = adata_raw.X
if sp.issparse(X_norm):
    X_norm = X_norm.toarray()

expr_df = pd.DataFrame(
    X_norm[:, [adata_raw.var_names.get_loc(g) for g in markers_to_check]],
    columns=markers_to_check,
    index=adata_raw.obs_names
)
expr_df['niche'] = adata_raw.obs[CLUSTER_KEY].values

# Mean expression per niche
mean_expr = expr_df.groupby('niche')[markers_to_check].mean().round(3)
print("Mean normalised expression per niche:")
print(mean_expr.to_string())

# Fraction of spots expressing each gene per niche (pct expressed)
pct_expr = expr_df.groupby('niche')[markers_to_check].apply(
    lambda x: (x > 0).mean()
).round(3)
print("\nFraction of spots expressing each gene per niche:")
print(pct_expr.to_string())

with the absolute expression we see that actually Niche 6 has the LOWEST immune marker expression of any niche and Niche 5 the highest. now we run a full marker check across all niches for the complete set of known prostate markers, and build the identity table from the ground up based on what the expression data actually shows rather than predictions

In [ ]:
# ── Comprehensive marker panel for prostate tissue niches ────────────

marker_panel = {
    # Epithelial
    'Luminal epithelial':  ['KLK3', 'KLK2', 'AR', 'NKX3-1', 'KRT8', 
                            'KRT18', 'SLC45A3', 'FXYD3'],
    'Basal epithelial':    ['KRT5', 'KRT14', 'TP63', 'CD44', 'BCAM'],
    'Club/Hillock':        ['PIGR', 'LTF', 'SCGB1A1', 'OLFM4', 'MMP7',
                            'CLDN4', 'LCN2'],
    'Neuroendocrine':      ['CHGA', 'SYP', 'ENO2', 'NCAM1'],
    
    # Stromal
    'Smooth muscle':       ['MYH11', 'ACTA2', 'TAGLN', 'DES', 'MYL9',
                            'TPM2'],
    'Fibroblast/CAF':      ['DCN', 'LUM', 'COL1A1', 'COL3A1', 'FAP',
                            'PDPN', 'POSTN', 'CLU', 'SPARCL1'],
    'Endothelial':         ['PECAM1', 'VWF', 'CDH5', 'CLDN5'],
    
    # Immune
    'B cell':              ['MS4A1', 'CD19', 'CD79A', 'CD79B', 'PAX5'],
    'Plasma cell':         ['JCHAIN', 'IGHG1', 'MZB1', 'XBP1', 'PRDM1'],
    'T cell general':      ['CD3D', 'CD3E', 'CD3G', 'CD2'],
    'T cytotoxic':         ['CD8A', 'CD8B', 'GZMB', 'GZMK', 'PRF1'],
    'T regulatory':        ['FOXP3', 'IL2RA', 'CTLA4', 'TIGIT'],
    'T helper':            ['CD4', 'IL7R', 'CCR7'],
    'Macrophage':          ['CD68', 'CD163', 'MRC1', 'SPP1', 'CSF1R'],
    'NK cell':             ['NCAM1', 'KLRD1', 'NKG7', 'GNLY'],
    'Dendritic':           ['ITGAX', 'CLEC9A', 'CD1C', 'FCER1A'],
    
    # Cancer specific
    'Prostate cancer':     ['TMPRSS2', 'ERG', 'AMACR', 'PCA3', 
                            'FOLH1', 'ACPP'],
    
    # Proliferation / quality
    'Proliferation':       ['MKI67', 'TOP2A', 'PCNA', 'CDK1'],
    'Mitochondrial':       ['MT-CO1', 'MT-CO2', 'MT-CO3', 'MT-ND1'],
    'Ribosomal':           ['RPS4X', 'RPL7A', 'RPS8', 'RPL26'],
}

# ── Compute mean expression per niche for all markers ────────────────
import scipy.sparse as sp

# Get all unique genes in panel that exist in dataset
all_markers = []
for genes in marker_panel.values():
    all_markers.extend(genes)
all_markers = list(dict.fromkeys(all_markers))  # deduplicate, preserve order

present = [g for g in all_markers if g in adata_raw.var_names]
absent  = [g for g in all_markers if g not in adata_raw.var_names]
print(f"Markers present in dataset: {len(present)}/{len(all_markers)}")
print(f"Absent: {absent}")

# Extract expression
X_norm = adata_raw.X
if sp.issparse(X_norm):
    X_norm = X_norm.toarray()

idx = [adata_raw.var_names.get_loc(g) for g in present]
expr_df = pd.DataFrame(
    X_norm[:, idx],
    columns=present,
    index=adata_raw.obs_names
)
expr_df['niche'] = adata_raw.obs[CLUSTER_KEY].values

mean_expr = expr_df.groupby('niche')[present].mean().round(3)
pct_expr  = expr_df.groupby('niche')[present].apply(
    lambda x: (x > 0).mean()
).round(3)

# ── Print results grouped by cell type category ──────────────────────
print("\n" + "="*80)
print("MEAN EXPRESSION PER NICHE — grouped by marker category")
print("="*80)

for category, genes in marker_panel.items():
    genes_present = [g for g in genes if g in present]
    if not genes_present:
        continue
    print(f"\n── {category} ──")
    print(mean_expr[genes_present].to_string())

# ── Find dominant marker category per niche ──────────────────────────
print("\n" + "="*80)
print("TOP EXPRESSED MARKER CATEGORY PER NICHE")
print("="*80)

for niche in sorted(mean_expr.index):
    category_scores = {}
    for category, genes in marker_panel.items():
        genes_present = [g for g in genes if g in present]
        if genes_present:
            category_scores[category] = mean_expr.loc[niche, genes_present].mean()
    
    top3 = sorted(category_scores.items(), key=lambda x: x[1], reverse=True)[:3]
    print(f"\nNiche {niche}:")
    for cat, score in top3:
        print(f"  {cat}: {score:.3f}")

# ── Save full results ─────────────────────────────────────────────────
mean_expr.to_csv(os.path.join(OUT_DIR, 'marker_panel_mean_expression.csv'))
pct_expr.to_csv(os.path.join(OUT_DIR, 'marker_panel_pct_expression.csv'))
print("\nSaved marker panel results to CSV.")